In [0]:
import json
import urllib.request
from typing import Dict

def send_teams_alert(alert_title: str, alert_message: str, webhook_url: str) -> str:
    """
    Sends a formatted alert message to a Microsoft Teams channel via a webhook.
    The AI agent should call this tool ONLY after investigating a pipeline failure 
    and generating a human-readable summary of the root cause.
    
    Args:
        alert_title: The heading of the alert (e.g., "Synergech Pipeline Failure: Driver Liability Assessment").
        alert_message: The detailed explanation of the issue, formatted by the AI Agent.
        webhook_url: The secure Microsoft Teams Webhook URL.
        
    Returns:
        A string confirming success or detailing the specific network/payload error.
    """
    payload = {
        "title": alert_title,
        "text": alert_message
    }
    
    data = json.dumps(payload).encode('utf-8')
    req = urllib.request.Request(
        webhook_url, 
        data=data, 
        headers={'Content-Type': 'application/json'}
    )
    
    try:
        with urllib.request.urlopen(req) as response:
            if response.status == 200:
                return f"Success: Alert delivered to Teams. HTTP Status: {response.status}"
            else:
                return f"Failure: Could not deliver alert. HTTP Status: {response.status}"
    except Exception as e:
        return f"System Error: Failed to execute webhook request. Details: {str(e)}"

In [0]:
%sql
CREATE OR REPLACE FUNCTION agent_monitoring.murugaveltarun.send_teams_alert(
  alert_title STRING COMMENT 'The heading of the alert.',
  alert_message STRING COMMENT 'The detailed explanation generated by the Agent.',
  webhook_url STRING COMMENT 'The secure Microsoft Teams Webhook URL.'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Sends a formatted alert message to a Microsoft Teams channel via a webhook. Called by the AI agent after investigating a pipeline failure.'
AS $$
  import json
  import urllib.request

  payload = {
      "title": alert_title,
      "text": alert_message
  }
  
  data = json.dumps(payload).encode('utf-8')
  req = urllib.request.Request(
      webhook_url, 
      data=data, 
      headers={'Content-Type': 'application/json'}
  )
  
  try:
      with urllib.request.urlopen(req) as response:
          if response.status == 200:
              return f"Success: Alert delivered. Status: {response.status}"
          else:
              return f"Failure: Could not deliver alert. Status: {response.status}"
  except Exception as e:
      return f"System Error: Details: {str(e)}"
$$;

In [0]:
# During Agent Execution
teams_webhook = dbutils.secrets.get(scope="synergech_monitoring", key="teams_webhook_url")

# The Agent is prompted to use this specific variable when invoking the tool

In [0]:
# 1. Clear the mismatched versions
%pip uninstall -y langchain langchain-core langchain-community databricks-langchain langchain-openai langgraph

# 2. Install the perfectly aligned 0.2.x family
%pip install langchain==0.2.16 langchain-core==0.2.43 langchain-community==0.2.16 langchain-openai==0.1.23 langgraph==0.2.16 databricks-langchain

# 3. Restart the Python engine
dbutils.library.restartPython()

Found existing installation: langchain 0.2.16
Uninstalling langchain-0.2.16:
  Successfully uninstalled langchain-0.2.16
Found existing installation: langchain-core 1.3.3
Uninstalling langchain-core-1.3.3:
  Successfully uninstalled langchain-core-1.3.3
Found existing installation: langchain-community 0.2.16
Uninstalling langchain-community-0.2.16:
  Successfully uninstalled langchain-community-0.2.16
Found existing installation: databricks-langchain 0.3.0
Uninstalling databricks-langchain-0.3.0:
  Successfully uninstalled databricks-langchain-0.3.0
Found existing installation: langchain-openai 1.2.1
Uninstalling langchain-openai-1.2.1:
  Successfully uninstalled langchain-openai-1.2.1
Found existing installation: langgraph 0.2.16
Uninstalling langgraph-0.2.16:
  Successfully uninstalled langgraph-0.2.16
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
  Using cached langchain-0.2.16-py3-none-any.whl.metadata (7.

In [0]:
%pip install -U langchain-openai
dbutils.library.restartPython()

  Using cached langchain_core-1.3.3-py3-none-any.whl.metadata (4.5 kB)
Using cached langchain_core-1.3.3-py3-none-any.whl (543 kB)
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.1.147
    Uninstalling langsmith-0.1.147:
      Successfully uninstalled langsmith-0.1.147
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.2.43
    Uninstalling langchain-core-0.2.43:
      Successfully uninstalled langchain-core-0.2.43
  Attempting uninstall: langchain-openai
    Found existing installation: langchain-openai 1.1.6
    Not uninstalling langchain-openai at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-d2d73981-c60c-4e2c-87ac-5778105c5c94
    Can't uninstall 'langchain-openai'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the followi

In [0]:
import os
from langchain_openai import AzureChatOpenAI
from langchain_community.tools.databricks import UCFunctionToolkit
from langgraph.prebuilt import create_react_agent

# 1. Fetch the secrets
teams_webhook = dbutils.secrets.get(scope="synergech_monitoring", key="teams_webhook_url")
azure_api_key = dbutils.secrets.get(scope="synergech_monitoring", key="azure_openai_key")

# 2. Set up Azure OpenAI Environment Variables
os.environ["AZURE_OPENAI_API_KEY"] = azure_api_key
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://synfra-agent-demo.services.ai.azure.com/" # <-- Update this!
os.environ["AZURE_OPENAI_API_VERSION"] = "2026-03-05" # Standard stable API version

# 3. Connect to Azure OpenAI
llm = AzureChatOpenAI(
    azure_deployment="gpt-5.4", # <-- Update this! (e.g., "gpt-4o")
    api_version="2024-02-01",
    temperature=0
)

# 4. Bind YOUR Unity Catalog Tool
toolkit = UCFunctionToolkit(
    warehouse_id="1a2b3661fe300d416eebc",  # <-- Make sure your Warehouse ID is still here!
    function_names=["agent_monitoring.murugaveltarun.send_teams_alert"]
)
tools = toolkit.get_tools()

# 5. The System Prompt 
system_prompt = f"""
You are the Synergech Pipeline Detective, an elite AI monitoring agent.
Your job is to analyze Databricks pipeline error logs and explain them in simple, non-technical terms.

When you receive an error log:
1. Identify the root cause of the failure.
2. Summarize the impact (e.g., "The Driver Liability Assessment failed to run").
3. Suggest a next step for the data engineering team.
4. ONLY after you have done this analysis, use your tool to send the alert to Microsoft Teams.

IMPORTANT SECURITY INSTRUCTION: 
When you invoke the send_teams_alert tool, you MUST use this exact webhook URL for the webhook_url parameter: {teams_webhook}
"""

# 6. Assemble the Agent using LangGraph
agent_executor = create_react_agent(llm, tools, state_modifier=system_prompt)

print("Agent successfully initialized using Azure OpenAI!")

Agent successfully initialized using Azure OpenAI!


In [0]:
# A simulated error log from your Driver Liability pipeline
messy_error_log = """
Py4JJavaError: An error occurred while calling o112.save.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 14.0 failed 4 times. 
Caused by: java.lang.NullPointerException: Value at column 'Driver_Age' in table 'raw_telemetry' cannot be null.
Constraint validation failed at synergech.liability.assessment.run(liability_calc.py:45)
"""

# Ask the agent to investigate (Notice the 'messages' format!)
print("Waking up Agent to investigate...")
response = agent_executor.invoke({
    "messages": [("user", f"Our pipeline just crashed. Here is the log: {messy_error_log}")]
})

print("\n--- Final Agent Output ---")
print(response["messages"][-1].content)

Waking up Agent to investigate...

--- Final Agent Output ---
The root cause is a null value in the `Driver_Age` column of the `raw_telemetry` table. The liability assessment step enforces a rule that `Driver_Age` cannot be empty, and that validation failed in `liability_calc.py` at line 45.

Impact: The Driver Liability Assessment failed to run, so downstream outputs from this pipeline step were not produced.

Recommended next step: The data engineering team should inspect the `raw_telemetry` input for records with missing `Driver_Age`, determine whether those records should be corrected, filtered, or defaulted, and then rerun the pipeline after fixing the data quality issue.
Root cause: A record in `raw_telemetry` has a missing `Driver_Age`, but the liability assessment requires that field to always be populated.

Impact: The Driver Liability Assessment failed and the pipeline stopped before producing its output.

Next step: Have the data engineering team identify and fix or exclude 